# Exploring the catalog inputs with inTRuder

What the TR locus catalog of §2 is built from, and what the finder makes of it. The
question driving this is the open one in
`submodules/inTRuder/src/python/intruder/pipeline/trf/params.py`: the `genome` parameter
profile is provisional, its values came from synthetic checks, and choosing them properly
means enumerating a grid rather than trusting a default.

**Sequence arrives through the pipeline, not through this notebook.** The cells below
launch `--stage preprocess` and read back the small tables it publishes. The genomes are
synced inside a task, streamed through the finder, and deleted there — so nothing here
holds a gigabyte of FASTA, and nothing downloads to wherever the notebook is running.
That is the same reason `SRA_RUN_METADATA` syncs and deletes the 2.3 GB metadata freeze
in one task.

Runs are cached: `-resume` means re-executing a cell costs nothing unless its parameters
changed.

```sh
pixi run -e notebooks lab
```

| Variable | Default | What it does |
| -------- | ------- | ------------ |
| `EDA_PROFILES` | `docker` | Nextflow profiles. `slurm,apptainer` on the cluster; empty for a stub run |
| `EDA_QUEUE` | unset | SLURM partition, e.g. `shared` |
| `EDA_STUB` | unset | `1` runs each task's stub: no S3, no finder, seconds not minutes |

In [ ]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Works whether the notebook is run from notebooks/ or from the repository root.
REPO = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file())
MANIFESTS = REPO / "manifests"
DATA = REPO / "data"
PUBLISHED = DATA / "derived" / "catalog"  # where the stage publishes; git-ignored

# Nextflow lives in its own pixi environment, so it is reached through pixi rather than
# imported. The notebooks environment does not carry it.
PIXI = shutil.which("pixi") or str(Path.home() / ".pixi" / "bin" / "pixi")

PROFILES = os.environ.get("EDA_PROFILES", "docker")
QUEUE = os.environ.get("EDA_QUEUE")
STUB = os.environ.get("EDA_STUB", "") == "1"

# Slots 1-3 of the project palette, the three that validate on all pairs.
SERIES = ("#2a78d6", "#eb6834", "#1baf7a")
INK = "#52514e"

plt.rcParams.update(
    {
        "figure.figsize": (8, 4),
        "figure.dpi": 120,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "grid.linewidth": 0.6,
        "axes.titlelocation": "left",
        "axes.titleweight": "semibold",
        "axes.labelcolor": INK,
        "xtick.color": INK,
        "ytick.color": INK,
    }
)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 120)

print(f"repo      {REPO}")
print(f"profiles  {PROFILES or '(none)'}{' queue ' + QUEUE if QUEUE else ''}")
print(f"stub      {STUB}")

## The driver

One function, because every data-producing cell below is the same call with different
parameters: run the stage, then read what it published. Nextflow's cache is the
memoisation — a second call with the same parameters resolves to `cached` and returns the
same file without re-syncing anything.

In [ ]:
def nextflow(preprocess: str, outputs: dict[str, str], **params) -> dict[str, Path]:
    """Run `--stage preprocess --preprocess <preprocess>` and return its published files.

    `outputs` maps a name to the published filename to read back. `params` become
    `--key value` on the command line, so a cell asks for a grid point by passing it.
    """
    cmd = [
        PIXI,
        "run",
        "-e",
        "nextflow",
        "nextflow",
        "run",
        ".",
        "--stage",
        "preprocess",
        "--preprocess",
        preprocess,
        "-ansi-log",
        "false",
        "-resume",
    ]
    # No profile is only useful with -stub: a real task needs an image to find the CLI
    # and the finder, which is the trap docs/workflows/preprocess/README.md names.
    if PROFILES:
        cmd += ["-profile", PROFILES]
    if QUEUE:
        cmd += ["--slurm_queue", QUEUE]
    if STUB:
        cmd += ["-stub"]
    for key, value in params.items():
        cmd += [f"--{key}", str(value)]

    done = subprocess.run(cmd, cwd=REPO, capture_output=True, text=True)
    print("\n".join(done.stdout.strip().splitlines()[-10:]))
    if done.returncode != 0:
        print(done.stderr.strip()[-2000:], file=sys.stderr)
        raise RuntimeError(f"nextflow exited {done.returncode}")

    resolved = {}
    for name, filename in outputs.items():
        path = PUBLISHED / filename
        if not path.is_file():
            raise FileNotFoundError(f"{path.relative_to(REPO)} was not published")
        resolved[name] = path
    return resolved

## 1. What the manifests say is available

`manifests/` is the inventory, not `data/`. A spec records what to fetch; its lockfile
records what was fetched, with a `sha256` per file only once the bytes were really
downloaded. So the lockfiles tell apart what exists upstream, what has ever been
downloaded, and what has only been listed. A spec with no lockfile has never been synced.

Committed JSON, so this needs no pipeline run.

In [ ]:
def inventory() -> pd.DataFrame:
    rows = []
    for spec_path in sorted(MANIFESTS.rglob("*.json")):
        if spec_path.name.endswith(".lock.json"):
            continue
        spec = spec_path.relative_to(MANIFESTS).with_suffix("").as_posix()
        lock_path = spec_path.with_name(f"{spec_path.stem}.lock.json")
        if not lock_path.is_file():
            rows.append(
                {
                    "spec": spec,
                    "files": None,
                    "GB": None,
                    "downloaded": None,
                    "state": "never synced",
                }
            )
            continue
        files = json.loads(lock_path.read_text())["files"]
        hashed = sum(1 for f in files if f["sha256"])
        rows.append(
            {
                "spec": spec,
                "files": len(files),
                "GB": round(sum(f["size"] for f in files) / 1e9, 3),
                "downloaded": hashed,
                "state": "listing only" if hashed == 0 else "downloaded",
            }
        )
    return pd.DataFrame(rows)


inv = inventory()
inv

In [ ]:
print("Never synced -- what §1 of docs/todo.md is still waiting on:")
for spec in inv.loc[inv.state == "never synced", "spec"]:
    print(f"  {spec}")
print()
print(f"Recorded upstream: {inv.GB.sum():.1f} GB over {int(inv.files.sum())} files")
print("The large body-site specs are 'listing only' on purpose: reads are streamed from")
print("S3 through a locus filter in §3, never stored. docs/data/HMP_pilot.md says why.")

## 2. The committed run tables

`data/derived/` is the one part of `data/` in git: tables the CLI extracted from the
metadata freeze, which was then deleted. Already built, so they load directly.

In [ ]:
runs = pd.read_csv(DATA / "derived" / "hmp-sra-runs.tsv", sep="\t")

# The filter docs/data/derived/hmp-sra-runs.md settles on. assay_type alone is not enough:
# 69 WGS runs are LS454, and SRP175119 is a 2019 crAssphage reanalysis of 22 samples that
# already have their original runs.
wgs = runs.query("assay_type == 'WGS' and platform == 'ILLUMINA' and study == 'SRP002163'")
print(f"{len(runs)} rows -> {len(wgs)} runs over {wgs['sample'].nunique()} samples")

per_sample = wgs.groupby("sample").size()
print(
    f"runs per sample: median {per_sample.median():.0f}, max {per_sample.max()}, "
    f"{(per_sample > 1).sum()} samples with more than one"
)
print("Depth is a covariate in §4, so these are pooled per sample before it is used.")
display(wgs["center"].value_counts().rename("runs").to_frame())

In [ ]:
counts = wgs["center"].value_counts()

fig, ax = plt.subplots(figsize=(7, 3))
bars = ax.barh(counts.index[::-1], counts.values[::-1], color=SERIES[0], height=0.6)
ax.bar_label(bars, padding=4, color=INK, fontsize=9)
ax.set_title("HMP WGS runs by sequencing center")
ax.set_xlabel("runs")
ax.set_xlim(0, counts.max() * 1.15)
ax.grid(axis="y", visible=False)
plt.tight_layout()
plt.show()

print(f"The top two centers hold {counts.head(2).sum() / counts.sum():.0%} of the runs.")
print("H2 asks whether genuine taxa cluster by center; with the design this unbalanced,")
print("the two small centers carry little weight whatever the clustering shows.")

In [ ]:
salter = pd.read_csv(DATA / "derived" / "salter-runs.tsv", sep="\t")
print(f"{len(salter)} runs")
display(
    salter.groupby("kit").agg(
        runs=("run", "size"), steps=("dilution", "nunique"), mbases=("mbases", "sum")
    )
)
print(f"{(salter.mbases == 0).sum()} runs report 0 Mbases, most of them CAMBIO steps 3-10.")
print("A step number is not comparable across kits, and depth falls with dilution by")
print("construction -- the confound H1 has to survive, not an inconvenience.")

## 3. The catalog, built on demand

`TR_CATALOG` syncs one genome manifest, runs `trf find` over every contig, writes the
catalog and its BED, and deletes the FASTAs. The mock community is the right spec to start
from: 22 known strains, 27 MB, and §2's last item is checking catalog loci against exactly
these genomes.

Two things in the task script are properties of how HMP distributes sequence rather than
of inTRuder, and both bite before any repeat finding happens:

- **The mock community is bzip2.** All 22 genomes are `.nuc.fsa.bz2`, and
  `trcore.io.read_fasta` is pyfastx, which reads plain and gzip only. The task runs
  `bunzip2` first.
- **`.fsa` is not sniffed.** `trf find --format auto` cannot place the extension, and the
  314 target-genus genomes are all `.fsa`, so the task passes `--format fasta` explicitly.

In [ ]:
out = nextflow(
    "catalog",
    {"catalog": "tr-catalog.tsv", "bed": "tr-catalog.bed"},
    catalog_spec=f"{MANIFESTS}/hmp/mock-community.json",
    catalog_genome_from="dir",
    catalog_profile="genome",
)
loci = pd.read_csv(out["catalog"], sep="\t")
print(f"\n{len(loci)} loci over {loci.genome.nunique()} genomes")
loci.head()

In [ ]:
per_genome = loci.groupby("genome").size().sort_values(ascending=False)
display(per_genome.rename("loci").to_frame())
print("A few dozen loci in a multi-megabase chromosome is the 'far too few' end of the")
print("failure params.py describes -- calls either way, just the wrong number of them.")
print("Section 5 measures that instead of eyeballing it.")

## 4. What the calls look like

Three columns decide whether a locus is usable downstream. Motif length separates classic
STRs from VNTRs, which behave differently under §3's stutter model. Purity says how far
the call drifted from an exact repeat. Repeat units say how much room there is for
allele-length variation at all — which is what §4 measures entropy over, so a locus with
few units has little to contribute whatever else is true of it.

In [ ]:
from intruder.pipeline.trf import params
from intruder.trcore import motifs

display(loci[["purity", "motif_length", "rep_length", "rep_units"]].describe().round(2))
print("\nmost common canonical motifs:")
display(loci.motif.map(motifs.canonical_motif).value_counts().head(10).rename("loci").to_frame())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
axes[0].hist(loci.motif_length, bins=range(1, int(loci.motif_length.max()) + 2), color=SERIES[0])
axes[0].set_title("Motif length")
axes[0].set_xlabel("bp")
axes[0].set_ylabel("loci")

gate = params.PROFILES["genome"].min_identity
axes[1].hist(loci.purity, bins=20, color=SERIES[1])
axes[1].set_title("Purity")
axes[1].set_xlabel("fraction of an exact repeat")
axes[1].axvline(gate, color=INK, lw=1.5, ls="--")
axes[1].annotate(
    f"min_identity {gate}",
    xy=(gate, 0),
    xytext=(5, 12),
    textcoords="offset points",
    color=INK,
    fontsize=9,
)
for ax in axes:
    ax.grid(axis="x", visible=False)
plt.tight_layout()
plt.show()

print("If purity piles up against the threshold, the gate is binding rather than slack:")
print("the surviving calls are the ones it let through, so its value is a real choice.")

## 5. The sweep: how much the thresholds decide

This is what the notebook is for. Three knobs decide whether a seed becomes a call —
`min_identity`, `min_seed_units`, `min_seed_length` — and `params.py` says the `genome`
profile's values are provisional. The question is what that costs.

`TR_SWEEP` is one task for the whole grid rather than `TR_CATALOG` once per point, because
a grid point is part of the task hash: a point per task would re-sync the same genomes for
every point. The genomes are the expensive part and the finder is cheap, so the loop goes
inside the task, where the sync is already paid for.

In [ ]:
out = nextflow(
    "sweep",
    {"sweep": "tr-sweep.tsv"},
    catalog_spec=f"{MANIFESTS}/hmp/mock-community.json",
    catalog_genome_from="dir",
    catalog_profile="genome",
    sweep_identities="0.8 0.7",
    sweep_seed_units="3 2",
    sweep_seed_lengths="10 8 6",
)
sweep = pd.read_csv(out["sweep"], sep="\t")
grid = sweep.groupby(
    ["min_identity", "min_seed_units", "min_seed_length"], as_index=False
).loci.sum()
display(grid.sort_values("loci"))
print(
    f"\nspan: {grid.loci.min()} to {grid.loci.max()} loci "
    f"({grid.loci.max() / max(grid.loci.min(), 1):.0f}x) over {len(grid)} grid points"
)

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.2))
for colour, (units, block) in zip(SERIES, grid.groupby("min_seed_units"), strict=False):
    for style, (identity, line) in zip(("-", "--"), block.groupby("min_identity"), strict=False):
        line = line.sort_values("min_seed_length", ascending=False)
        ax.plot(
            line.min_seed_length,
            line.loci,
            style,
            color=colour,
            lw=2,
            marker="o",
            ms=6,
            label=f"{units} seed units, identity {identity}",
        )
ax.set_yscale("log")
ax.set_title("Loci found across the threshold grid")
ax.set_xlabel("min_seed_length (bp) — looser to the right")
ax.set_ylabel("loci, all genomes (log scale)")
ax.invert_xaxis()
ax.legend(frameon=False, fontsize=9, labelcolor=INK)
plt.tight_layout()
plt.show()

print("The log axis is the finding, not a presentation choice: on a linear axis the")
print("strictest points are indistinguishable from zero. Nothing downstream is stable")
print("across a range this wide -- §4 computes entropy per locus, so the number of loci")
print("sets what there is to measure at all.")
print()
print("The repo's answer is in the §2 todo: choose with novelty.search in enumeration")
print("mode, whose `agreement` objective needs no labelled loci. This is the surface")
print("that decision gets made over.")

## 6. What counts as one locus

Two things make the raw call list an overcount, and both have to be settled before
cross-sample entropy means anything: `pytrf` reports overlapping calls over the same
stretch, and six genomes per genus repeat near-identical loci.

The §2 plan merges on canonical motif plus interval overlap, so the merge rule and the
screen's match rule are one rule. `trcore.motifs.canonical_motif` is that key — but its
default policy folds rotation and **not** reverse complement, which for a bacterial
catalog is a decision rather than a detail: a repeat on the minus strand is the same locus.

In [ ]:
rotation_only = motifs.MotifEquivalence(circular=True, reverse_complement=False)
with_strand = motifs.MotifEquivalence(circular=True, reverse_complement=True)

display(
    pd.DataFrame(
        [
            {
                "motif": m,
                "rotation only": motifs.canonical_motif(m, rotation_only),
                "with reverse complement": motifs.canonical_motif(m, with_strand),
            }
            for m in ("CGG", "GGC", "GCG", "CCG", "AT", "TA", "AAT", "ATT")
        ]
    )
)
print("CGG and CCG are the same repeat read from opposite strands: two catalog rows under")
print("the default policy, one under reverse_complement=True.")
print()
for policy, name in ((rotation_only, "rotation only"), (with_strand, "with reverse complement")):
    keyed = loci.motif.map(lambda m, policy=policy: motifs.canonical_motif(m, policy))
    print(f"  {name:26s} -> {keyed.nunique():5d} distinct motifs across {len(loci)} calls")

In [ ]:
ordered = loci.sort_values(["genome", "contig", "start"])
same_contig = (ordered.contig == ordered.contig.shift()) & (
    ordered.genome == ordered.genome.shift()
)
overlapping = ((ordered.start < ordered.end.shift()) & same_contig).sum()
print(f"{overlapping} of {len(loci)} calls start before the previous call on the same")
print("contig ends. Those are what an interval merge collapses; the motif key decides")
print("which of them are the same repeat rather than two repeats in one place.")

## What this leaves

Settled: the catalog runs end to end through the pipeline, nothing stores bulk sequence,
and the two HMP format traps are handled in the task rather than worked around by hand.

Not settled, in the order §2 needs them:

1. **The thresholds.** Section 5 gives the sensitivity surface; choosing a point on it is
   `novelty.search` in enumeration mode, which is not wired up yet.
2. **The merge.** Section 6 shows both halves — the motif key and the overlaps — but no
   merge step exists, so every count above is of raw calls.
3. **The genomes that matter.** This ran against 22 mock strains. The catalog the project
   needs is the 198 target-genus genomes and the contaminant genera, and three of those
   specs have never been synced (section 1).

For a different spec, the same call with `catalog_spec` pointed elsewhere and
`catalog_genome_from='file'` for the slacken layouts, which put one genome per file
rather than one per directory:

```python
out = nextflow("catalog", {"catalog": "tr-catalog.tsv", "bed": "tr-catalog.bed"},
               catalog_spec=f"{MANIFESTS}/contaminants/kit.json",
               catalog_genome_from="file")
```